# Instacart 2017 — Market Basket & Customer Behavior Analysis

This notebook builds on the [Instacart Online Grocery Shopping Dataset 2017](https://www.instacart.com/datasets/grocery-shopping-2017) to perform:

1. **Market basket analysis** — finding products frequently bought together using association-rule mining (Apriori algorithm via `mlxtend`).
2. **Customer behavior analysis** — segmenting users by ordering frequency, basket size, loyalty, and temporal patterns.

All visualizations use **Plotly Express** with the `simple_white` template.

In [ ]:
import os
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

px.defaults.template = "simple_white"


## 1 — Load the Data

In [ ]:
if not os.path.exists("orders.csv"):
    with zipfile.ZipFile("orders.csv.zip", "r") as z:
        z.extractall(".")

orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")
aisles = pd.read_csv("aisles.csv")
departments = pd.read_csv("departments.csv")
order_products = pd.read_csv("order_products_train.csv")

# Enriched order-products table
op = (
    order_products
    .merge(products, on="product_id")
    .merge(aisles, on="aisle_id")
    .merge(departments, on="department_id")
)

# Link order metadata
op_full = op.merge(orders[["order_id", "user_id", "order_dow", "order_hour_of_day",
                           "order_number", "days_since_prior_order"]], on="order_id")

print(f"orders:          {orders.shape}")
print(f"order_products:  {order_products.shape}")
print(f"products:        {products.shape}")
print(f"enriched op:     {op.shape}")


---
# Part A — Market Basket Analysis

We identify products that are frequently purchased together using co-occurrence counting and formal association-rule mining (Apriori).

## 2 — Top 15 Most Frequently Co-Purchased Product Pairs

In [ ]:
from itertools import combinations

# Build baskets: list of product names per order (limit to top 100 products for speed)
top100 = op["product_name"].value_counts().head(100).index
op_top = op[op["product_name"].isin(top100)]

baskets = op_top.groupby("order_id")["product_name"].apply(set)

# Count co-occurrences
pair_counts = {}
for basket in baskets:
    if len(basket) < 2:
        continue
    for pair in combinations(sorted(basket), 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

pairs_df = (
    pd.DataFrame(
        [(p[0], p[1], c) for p, c in pair_counts.items()],
        columns=["Product A", "Product B", "Co-occurrence Count"],
    )
    .sort_values("Co-occurrence Count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

pairs_df["Pair"] = pairs_df["Product A"] + "  +  " + pairs_df["Product B"]

fig = px.bar(
    pairs_df,
    x="Co-occurrence Count",
    y="Pair",
    orientation="h",
    title="Top 15 Most Frequently Co-Purchased Product Pairs",
    text_auto=True,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


## 3 — Top 15 Most Frequently Co-Purchased Aisle Pairs

In [ ]:
aisle_baskets = op.groupby("order_id")["aisle"].apply(set)

aisle_pair_counts = {}
for basket in aisle_baskets:
    if len(basket) < 2:
        continue
    for pair in combinations(sorted(basket), 2):
        aisle_pair_counts[pair] = aisle_pair_counts.get(pair, 0) + 1

aisle_pairs_df = (
    pd.DataFrame(
        [(p[0], p[1], c) for p, c in aisle_pair_counts.items()],
        columns=["Aisle A", "Aisle B", "Co-occurrence Count"],
    )
    .sort_values("Co-occurrence Count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

aisle_pairs_df["Pair"] = aisle_pairs_df["Aisle A"] + "  +  " + aisle_pairs_df["Aisle B"]

fig = px.bar(
    aisle_pairs_df,
    x="Co-occurrence Count",
    y="Pair",
    orientation="h",
    title="Top 15 Most Frequently Co-Purchased Aisle Pairs",
    text_auto=True,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


## 4 — Association Rules (Apriori Algorithm)

We use the Apriori algorithm from `mlxtend` on aisles to find association rules with meaningful support, confidence, and lift.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Build aisle-level baskets (faster and more interpretable than product-level)
aisle_basket_list = op.groupby("order_id")["aisle"].apply(list).tolist()

te = TransactionEncoder()
te_array = te.fit(aisle_basket_list).transform(aisle_basket_list)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

# Run Apriori with a reasonable minimum support
frequent_items = apriori(basket_df, min_support=0.02, use_colnames=True)
print(f"Frequent itemsets found: {len(frequent_items)}")

# Generate association rules
rules = association_rules(frequent_items, metric="lift", min_threshold=1.0)
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)
print(f"Association rules found: {len(rules)}")
rules.head(10)


## 5 — Top 20 Association Rules by Lift

In [ ]:
top_rules = rules.head(20).copy()
top_rules["antecedents_str"] = top_rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
top_rules["consequents_str"] = top_rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
top_rules["Rule"] = top_rules["antecedents_str"] + "  →  " + top_rules["consequents_str"]

fig = px.bar(
    top_rules,
    x="lift",
    y="Rule",
    orientation="h",
    title="Top 20 Association Rules by Lift",
    text_auto=".2f",
    hover_data=["support", "confidence"],
)
fig.update_layout(yaxis=dict(autorange="reversed"), xaxis_title="Lift")
fig.show()


## 6 — Support vs Confidence of Association Rules

In [ ]:
rules_viz = rules.copy()
rules_viz["antecedents_str"] = rules_viz["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules_viz["consequents_str"] = rules_viz["consequents"].apply(lambda x: ", ".join(sorted(x)))
rules_viz["Rule"] = rules_viz["antecedents_str"] + " → " + rules_viz["consequents_str"]

fig = px.scatter(
    rules_viz,
    x="support",
    y="confidence",
    size="lift",
    color="lift",
    hover_name="Rule",
    title="Association Rules — Support vs Confidence (size & color = Lift)",
    labels={"support": "Support", "confidence": "Confidence", "lift": "Lift"},
    color_continuous_scale="Viridis",
)
fig.show()


---
# Part B — Customer Behavior Analysis

We segment and profile customers based on their ordering history.

## 7 — Build Customer Profiles

In [ ]:
# Per-user aggregations from the full orders table
user_orders = orders.groupby("user_id").agg(
    total_orders=("order_id", "nunique"),
    avg_days_between_orders=("days_since_prior_order", "mean"),
    avg_dow=("order_dow", "mean"),
    avg_hour=("order_hour_of_day", "mean"),
).reset_index()

# Basket-size stats from train set
basket_sizes = op_full.groupby(["user_id", "order_id"]).size().reset_index(name="basket_size")
user_basket = basket_sizes.groupby("user_id")["basket_size"].mean().reset_index(name="avg_basket_size")

# Reorder rate per customer
user_reorder = op_full.groupby("user_id")["reordered"].mean().reset_index(name="reorder_rate")

# Merge
customers = user_orders.merge(user_basket, on="user_id", how="left").merge(user_reorder, on="user_id", how="left")
customers.head()


## 8 — Distribution of Customer Lifetime Orders

In [ ]:
fig = px.histogram(
    customers,
    x="total_orders",
    nbins=100,
    title="Distribution of Total Orders per Customer",
    labels={"total_orders": "Total Orders", "count": "Number of Customers"},
)
fig.show()


## 9 — Distribution of Average Basket Size per Customer

In [ ]:
fig = px.histogram(
    customers.dropna(subset=["avg_basket_size"]),
    x="avg_basket_size",
    nbins=50,
    title="Distribution of Average Basket Size per Customer",
    labels={"avg_basket_size": "Avg Items per Order", "count": "Number of Customers"},
)
fig.show()


## 10 — Distribution of Customer Reorder Rates

In [ ]:
fig = px.histogram(
    customers.dropna(subset=["reorder_rate"]),
    x="reorder_rate",
    nbins=50,
    title="Distribution of Customer Reorder Rates",
    labels={"reorder_rate": "Reorder Rate", "count": "Number of Customers"},
)
fig.show()


## 11 — Customer Segments by Order Frequency

We split customers into groups based on how many total orders they have placed.

In [ ]:
def frequency_segment(n):
    if n <= 5:
        return "Light (1-5)"
    elif n <= 15:
        return "Regular (6-15)"
    elif n <= 30:
        return "Frequent (16-30)"
    else:
        return "Power (31+)"

customers["frequency_segment"] = customers["total_orders"].apply(frequency_segment)

seg_order = ["Light (1-5)", "Regular (6-15)", "Frequent (16-30)", "Power (31+)"]
seg_counts = (
    customers["frequency_segment"]
    .value_counts()
    .reindex(seg_order)
    .reset_index()
)
seg_counts.columns = ["Segment", "Number of Customers"]

fig = px.bar(
    seg_counts,
    x="Segment",
    y="Number of Customers",
    title="Customer Segments by Order Frequency",
    text_auto=True,
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()


## 12 — Average Metrics by Customer Segment

In [ ]:
seg_profile = (
    customers
    .groupby("frequency_segment")
    .agg(
        num_customers=("user_id", "count"),
        avg_orders=("total_orders", "mean"),
        avg_basket_size=("avg_basket_size", "mean"),
        avg_reorder_rate=("reorder_rate", "mean"),
        avg_days_between=("avg_days_between_orders", "mean"),
    )
    .reindex(seg_order)
    .reset_index()
)
seg_profile.columns = ["Segment", "Customers", "Avg Orders", "Avg Basket Size",
                        "Avg Reorder Rate", "Avg Days Between Orders"]
seg_profile


## 13 — Average Reorder Rate by Customer Segment

In [ ]:
fig = px.bar(
    seg_profile,
    x="Segment",
    y="Avg Reorder Rate",
    title="Average Reorder Rate by Customer Segment",
    text_auto=".2f",
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()


## 14 — Average Days Between Orders by Customer Segment

In [ ]:
fig = px.bar(
    seg_profile,
    x="Segment",
    y="Avg Days Between Orders",
    title="Average Days Between Orders by Customer Segment",
    text_auto=".1f",
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()


## 15 — Customer Ordering Patterns: Day of Week × Hour of Day

A heatmap showing when customers place their orders.

In [ ]:
dow_map = {0: "Saturday", 1: "Sunday", 2: "Monday", 3: "Tuesday",
           4: "Wednesday", 5: "Thursday", 6: "Friday"}
orders_heatmap = orders.copy()
orders_heatmap["day_name"] = orders_heatmap["order_dow"].map(dow_map)

heatmap_data = (
    orders_heatmap
    .groupby(["day_name", "order_hour_of_day"])
    .size()
    .reset_index(name="count")
)

# Pivot for heatmap
pivot = heatmap_data.pivot(index="day_name", columns="order_hour_of_day", values="count")
pivot = pivot.reindex([dow_map[i] for i in range(7)])

fig = px.imshow(
    pivot,
    title="Order Volume by Day of Week and Hour of Day",
    labels=dict(x="Hour of Day", y="Day of Week", color="Orders"),
    aspect="auto",
    color_continuous_scale="Blues",
)
fig.show()


## 16 — Basket Size vs Reorder Rate

In [ ]:
# Bin basket sizes and compute average reorder rate per bin
customers_clean = customers.dropna(subset=["avg_basket_size", "reorder_rate"])
customers_clean = customers_clean.copy()
customers_clean["basket_bin"] = pd.cut(customers_clean["avg_basket_size"], bins=20)
basket_reorder = (
    customers_clean
    .groupby("basket_bin", observed=True)["reorder_rate"]
    .mean()
    .reset_index()
)
basket_reorder["basket_bin_mid"] = basket_reorder["basket_bin"].apply(lambda x: x.mid)

fig = px.line(
    basket_reorder,
    x="basket_bin_mid",
    y="reorder_rate",
    title="Average Reorder Rate by Basket Size",
    labels={"basket_bin_mid": "Average Basket Size", "reorder_rate": "Avg Reorder Rate"},
    markers=True,
)
fig.show()


## 17 — Order Frequency vs Days Between Orders

In [ ]:
cust_sample = customers.dropna(subset=["avg_days_between_orders", "avg_basket_size"]).sample(
    n=min(5000, len(customers)), random_state=42
)

fig = px.scatter(
    cust_sample,
    x="total_orders",
    y="avg_days_between_orders",
    color="frequency_segment",
    size="avg_basket_size",
    title="Order Frequency vs Average Days Between Orders (sample of 5,000 customers)",
    labels={
        "total_orders": "Total Orders",
        "avg_days_between_orders": "Avg Days Between Orders",
        "avg_basket_size": "Avg Basket Size",
        "frequency_segment": "Segment",
    },
    color_discrete_sequence=px.colors.qualitative.Set2,
    opacity=0.6,
    category_orders={"frequency_segment": seg_order},
)
fig.show()


## 18 — Department Preferences by Customer Segment

In [ ]:
# Map user → segment
user_seg = customers[["user_id", "frequency_segment"]].copy()

# Join with op_full to get department per item per user
seg_dept = op_full.merge(user_seg, on="user_id")
seg_dept_counts = (
    seg_dept
    .groupby(["frequency_segment", "department"])
    .size()
    .reset_index(name="item_count")
)

# Normalize within each segment to get proportions
seg_totals = seg_dept_counts.groupby("frequency_segment")["item_count"].transform("sum")
seg_dept_counts["proportion"] = seg_dept_counts["item_count"] / seg_totals

# Keep top 10 departments overall for readability
top10_depts = op["department"].value_counts().head(10).index.tolist()
seg_dept_top = seg_dept_counts[seg_dept_counts["department"].isin(top10_depts)]

fig = px.bar(
    seg_dept_top,
    x="frequency_segment",
    y="proportion",
    color="department",
    title="Department Proportion by Customer Segment (Top 10 Departments)",
    labels={"frequency_segment": "Customer Segment", "proportion": "Proportion of Items",
            "department": "Department"},
    barmode="stack",
    category_orders={"frequency_segment": seg_order},
)
fig.show()


## 19 — How Reorder Rate Evolves Over Customer Lifetime

In [ ]:
# For orders in train set, link order_number
op_lifetime = order_products.merge(
    orders[["order_id", "order_number"]], on="order_id"
)

reorder_by_order_num = (
    op_lifetime
    .groupby("order_number")["reordered"]
    .mean()
    .reset_index()
)
reorder_by_order_num.columns = ["Order Number", "Reorder Rate"]
# Limit to first 50 orders for readability
reorder_by_order_num = reorder_by_order_num[reorder_by_order_num["Order Number"] <= 50]

fig = px.line(
    reorder_by_order_num,
    x="Order Number",
    y="Reorder Rate",
    title="Reorder Rate by Order Number in Customer Lifetime",
    markers=True,
)
fig.show()
